In [1]:
##############################################
##############################################
### TO GET ACCESS TO FILE IN GOOGLE COLAB ###
##############################################
##############################################

# To get access to the data files from github in Google Colab
!git clone https://github.com/math869c/graph_representation_st457.git

# set the folder to get access to the data
import os
os.chdir('/content') # to avoid error if rerun
os.chdir('./graph_representation_st457') # gives them access to everything in the folder

Cloning into 'graph_representation_st457'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 103 (delta 45), reused 73 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 26.31 MiB | 9.47 MiB/s, done.
Resolving deltas: 100% (45/45), done.


In [4]:
#!pip install optuna

In [5]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch
import torch.optim as optim
import json
import optuna
import time

# load from other folders
from helper_functions import *
from model_classes import *

In [6]:
# Stock prices
open_prices_interp = pd.read_csv('data_folder/open_prices_interp.csv', index_col=0)
# into numpy
x = open_prices_interp.to_numpy()
tickers_with_data = open_prices_interp.columns

# Load graph data
with open("data_folder/firm_industry.json", "r") as f:
    firm_industry_dict = json.load(f)
A = np.load("data_folder/adjacency_matrix.npy")
adj_matrix = torch.tensor(A, dtype=torch.float32)

# make training data
batch_size =32
X_train, y_train, X_val, y_val, X_test, y_test, sc, train_loader, val_loader, test_loader = create_data(x,
                                                                                                        batch_size =batch_size,
                                                                                                        flatten_data = True, # Should be True for LSTM, False for GTC and GAT
                                                                                                        flatten_time_features=False # Should be True for GAT, False for LSTM and GTC
                                                                                                        )

torch.Size([984, 8, 2300])
torch.Size([984, 460])


# building optuna objective function for hyperparameter tuning

In [7]:
SEARCH_N_TRIALS = 30

best_model_state_global = None
best_loss_global = float("inf")

def objective(trial):
    global best_model_state_global, best_loss_global

    hidden_units = trial.suggest_int("lstm_units", 64, 256, step=64)
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 5e-4, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])
    epochs = trial.suggest_int("epochs", 15, 40)
    patience = trial.suggest_int("patience", 5, 10)

    model = LSTMRegressor(
        input_size=X_train.shape[2],
        hidden_units=hidden_units,
        output_size=y_train.shape[1],
        dropout_rate=dropout_rate
    )

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_val_loss = float("inf")
    best_state_dict = None
    patience_counter = 0
    trial_start_time = time.time()

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch_LSTM(model, train_loader, criterion, optimizer)
        val_loss, val_acc = evaluate_LSTM(model, val_loader, criterion)

        print(
            f"Trial {trial.number + 1}/{SEARCH_N_TRIALS} | "
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} Val Acc: {val_acc:.4f}"
        )

        trial.report(val_loss, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state_dict = copy.deepcopy(model.state_dict())
            patience_counter = 0
            trial.set_user_attr("best_val_acc", val_acc)
            if val_loss < best_loss_global:
                best_loss_global = val_loss
                best_model_state_global = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    trial.set_user_attr("training_time_s", time.time() - trial_start_time)
    return best_val_loss


# Fine tune model

In [8]:
# run trials
best_overall_loss, best_config, best_model_final, best_history_final = 100.0, {}, None, None

results_log = []
search_start_time = time.time()

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=SEARCH_N_TRIALS)

for trial in study.trials:
    if trial.value is None:
        continue

    results_log.append({
        'config_id': trial.number + 1,
        'params': str(trial.params),
        'val_loss': trial.value,
        'training_time_s': trial.user_attrs.get('training_time_s', np.nan)
    })

results_df = pd.DataFrame(results_log).sort_values(by='val_loss', ascending=True)
search_end_time = time.time()
total_duration = search_end_time - search_start_time

best_params = study.best_trial.params
best_config = dict(best_params)
best_overall_loss = study.best_trial.value

print("\n" + "=" * 60)
print(f"Optuna Search Complete in {total_duration:.2f} seconds.")
print(f"Best Configuration: {best_config} | Best Val loss: {best_overall_loss:.4f}")

print("\n--- Detailed Hyperparameter Search Results ---")
print(results_df)

print("\nBest trial:")
print("  Value:", study.best_trial.value)
print("  Params:")
for k, v in study.best_trial.params.items():
    print(f"    {k}: {v}")


[I 2026-04-19 12:52:09,514] A new study created in memory with name: no-name-84dfd541-a3b5-42e4-ae1c-abcbd640a293


Trial 1/30 | Epoch 1/36 | Train Loss: 1.0087 Acc: 0.5013 | Val Loss: 2.6169 Val Acc: 0.4998
Trial 1/30 | Epoch 2/36 | Train Loss: 1.0058 Acc: 0.5023 | Val Loss: 2.6154 Val Acc: 0.5006
Trial 1/30 | Epoch 3/36 | Train Loss: 1.0046 Acc: 0.5043 | Val Loss: 2.6146 Val Acc: 0.5012
Trial 1/30 | Epoch 4/36 | Train Loss: 1.0030 Acc: 0.5096 | Val Loss: 2.6144 Val Acc: 0.5043
Trial 1/30 | Epoch 5/36 | Train Loss: 1.0003 Acc: 0.5170 | Val Loss: 2.6166 Val Acc: 0.5047
Trial 1/30 | Epoch 6/36 | Train Loss: 0.9928 Acc: 0.5395 | Val Loss: 2.6215 Val Acc: 0.4990
Trial 1/30 | Epoch 7/36 | Train Loss: 0.9842 Acc: 0.5479 | Val Loss: 2.6247 Val Acc: 0.5131
Trial 1/30 | Epoch 8/36 | Train Loss: 0.9804 Acc: 0.5558 | Val Loss: 2.6289 Val Acc: 0.5031


[I 2026-04-19 12:52:29,356] Trial 0 finished with value: 2.614380159685689 and parameters: {'lstm_units': 64, 'dropout_rate': 0.4812048835685762, 'learning_rate': 0.0008552950761731467, 'batch_size': 32, 'epochs': 36, 'patience': 5}. Best is trial 0 with value: 2.614380159685689.


Trial 1/30 | Epoch 9/36 | Train Loss: 0.9592 Acc: 0.5764 | Val Loss: 2.6305 Val Acc: 0.5090
Early stopping triggered.
Trial 2/30 | Epoch 1/38 | Train Loss: 1.0081 Acc: 0.4993 | Val Loss: 2.6162 Val Acc: 0.5018
Trial 2/30 | Epoch 2/38 | Train Loss: 1.0054 Acc: 0.5016 | Val Loss: 2.6150 Val Acc: 0.5010
Trial 2/30 | Epoch 3/38 | Train Loss: 1.0028 Acc: 0.5072 | Val Loss: 2.6147 Val Acc: 0.5013
Trial 2/30 | Epoch 4/38 | Train Loss: 0.9983 Acc: 0.5225 | Val Loss: 2.6184 Val Acc: 0.5044
Trial 2/30 | Epoch 5/38 | Train Loss: 0.9876 Acc: 0.5417 | Val Loss: 2.6257 Val Acc: 0.5018
Trial 2/30 | Epoch 6/38 | Train Loss: 0.9716 Acc: 0.5567 | Val Loss: 2.6258 Val Acc: 0.5062
Trial 2/30 | Epoch 7/38 | Train Loss: 0.9555 Acc: 0.5755 | Val Loss: 2.6382 Val Acc: 0.5052
Trial 2/30 | Epoch 8/38 | Train Loss: 0.9479 Acc: 0.5817 | Val Loss: 2.6621 Val Acc: 0.5107
Trial 2/30 | Epoch 9/38 | Train Loss: 0.9233 Acc: 0.5992 | Val Loss: 2.6909 Val Acc: 0.5070


[I 2026-04-19 12:52:44,425] Trial 1 finished with value: 2.614735953269466 and parameters: {'lstm_units': 64, 'dropout_rate': 0.32900819963889383, 'learning_rate': 0.0007262309230220564, 'batch_size': 64, 'epochs': 38, 'patience': 7}. Best is trial 0 with value: 2.614380159685689.


...

Trial 28/30 | Epoch 8/40 | Train Loss: 0.8807 Acc: 0.6282 | Val Loss: 2.7484 Val Acc: 0.5077
Early stopping triggered.
Trial 29/30 | Epoch 1/26 | Train Loss: 1.0056 Acc: 0.5015 | Val Loss: 2.6139 Val Acc: 0.5031
Trial 29/30 | Epoch 2/26 | Train Loss: 1.0021 Acc: 0.5115 | Val Loss: 2.6136 Val Acc: 0.5081
Trial 29/30 | Epoch 3/26 | Train Loss: 0.9971 Acc: 0.5279 | Val Loss: 2.6146 Val Acc: 0.5129
Trial 29/30 | Epoch 4/26 | Train Loss: 0.9860 Acc: 0.5494 | Val Loss: 2.6229 Val Acc: 0.5006
Trial 29/30 | Epoch 5/26 | Train Loss: 0.9610 Acc: 0.5709 | Val Loss: 2.6339 Val Acc: 0.5069
Trial 29/30 | Epoch 6/26 | Train Loss: 0.9383 Acc: 0.5884 | Val Loss: 2.6399 Val Acc: 0.5039
Trial 29/30 | Epoch 7/26 | Train Loss: 0.9037 Acc: 0.6150 | Val Loss: 2.6943 Val Acc: 0.5033
Trial 29/30 | Epoch 8/26 | Train Loss: 0.8680 Acc: 0.6305 | Val Loss: 2.8979 Val Acc: 0.4988
Trial 29/30 | Epoch 9/26 | Train Loss: 0.8268 Acc: 0.6459 | Val Loss: 2.7530 Val Acc: 0.5030
Trial 29/30 | Epoch 10/26 | Train Loss: 0.81

[I 2026-04-19 13:11:44,906] Trial 28 finished with value: 2.613611271304469 and parameters: {'lstm_units': 192, 'dropout_rate': 0.21711668759850206, 'learning_rate': 0.0005373767416380328, 'batch_size': 64, 'epochs': 26, 'patience': 9}. Best is trial 8 with value: 2.6130745949283725.


Trial 29/30 | Epoch 11/26 | Train Loss: 0.7865 Acc: 0.6597 | Val Loss: 2.7740 Val Acc: 0.5014
Early stopping triggered.


[I 2026-04-19 13:11:46,274] Trial 29 pruned. 


Trial 30/30 | Epoch 1/24 | Train Loss: 1.0088 Acc: 0.5006 | Val Loss: 2.6165 Val Acc: 0.5012

Optuna Search Complete in 1176.78 seconds.
Best Configuration: {'lstm_units': 128, 'dropout_rate': 0.4641790920735855, 'learning_rate': 0.0008481129576641046, 'batch_size': 32, 'epochs': 16, 'patience': 8} | Best Val loss: 2.6131

--- Detailed Hyperparameter Search Results ---
    config_id                                             params  val_loss  \
8           9  {'lstm_units': 128, 'dropout_rate': 0.46417909...  2.613075   
9          10  {'lstm_units': 256, 'dropout_rate': 0.29117456...  2.613467   
5           6  {'lstm_units': 256, 'dropout_rate': 0.42118495...  2.613468   
22         23  {'lstm_units': 256, 'dropout_rate': 0.41881952...  2.613474   
17         18  {'lstm_units': 192, 'dropout_rate': 0.44599034...  2.613485   
21         22  {'lstm_units': 256, 'dropout_rate': 0.41403166...  2.613564   
20         21  {'lstm_units': 256, 'dropout_rate': 0.46255541...  2.613588   
10  

# Save best model

In [9]:
output_size = y_train.shape[1]
input_size = X_train.shape[2]

model_LSTM = LSTMRegressor(
    input_size=input_size,
    hidden_units=best_params['lstm_units'],
    output_size=output_size,
    dropout_rate=best_params['dropout_rate']
)
model_LSTM.load_state_dict(best_model_state_global)

criterion = nn.MSELoss()
optimizer = optim.Adam(model_LSTM.parameters(), lr=best_params['learning_rate'])
MODEL_SAVE_PATH = 'best_lstm_model_graphtheory.pt'

torch.save({
    'model_state_dict': model_LSTM.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'hyperparameters': best_params,
}, MODEL_SAVE_PATH)

In [11]:
from google.colab import files
files.download(MODEL_SAVE_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
test_loss_LSTM, _ = evaluate_LSTM(model_LSTM, test_loader, criterion)
print("Test loss:", test_loss_LSTM)


Test loss: 1.5549269953081686
